# FLAG 2027 - Notebook 23: bigger CLIP-family face encoders (stage 1: challenge + all MAV, no VoxCeleb)

Why: cross-modal matching uses coarse appearance attributes, not fine identity (our ArcFace result; the FAME 2026 runner-up found the same: ImageBind/OpenCLIP ViT-H 24.7 EER vs ArcFace 37-48). We use CLIP ViT-L/14. Candidates, all frozen, all public:

| Name | Model | HF id | dim |
|---|---|---|---|
| `vith` | OpenCLIP ViT-H/14 laion2b (= ImageBind vision tower, used by FAME 2026 #2) | laion/CLIP-ViT-H-14-laion2B-s32B-b79K | 1024 |
| `dfn` | Apple DFN-5B ViT-H/14 @378 px | apple/DFN5B-CLIP-ViT-H-14-378 | 1024 |
| `pe` | Meta Perception Encoder Core L/14 @336 px | timm/PE-Core-L-14-336 | 1024 |
| `farl` | FaRL ViT-B/16 (face-specific CLIP, 20M face-caption pairs) | Green-Sky/FaRL-Base-Patch16-LAIONFace20M-ep64 | 512 |

Steps: face file lists in our row order (alignment check against cached CLIP features) -> extract the four encoders -> CV: each one REPLACING CLIP, then CLIP + the best one concatenated -> ZIP of the best (no VoxCeleb). Voice side = R4 stack, same training as NB21 (reference S0 = 25.53 is reused from NB21).

About 2.5-3 h. Resumable. Paste the **SUMMARY**.

In [ ]:
# Cell 1 - Drive, GPU, packages
from google.colab import drive
drive.mount('/content/drive')
!pip -q install open_clip_torch timm
import torch, shutil
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print('[v1 28-Oct] Device:', dev, '| disk free %.0f GB' % (shutil.disk_usage('/content')[2] / 1e9))

In [ ]:
# Cell 2 - FACE file lists in the SAME row order as our feature files (challenge, MAV v1/v3, MAV extra) + alignment check with cached CLIP
import os, re, sys, json, time, zipfile, copy
import numpy as np, pandas as pd
from PIL import Image
from tqdm.auto import tqdm
BASE = '/content/drive/MyDrive'
PROJECT = os.path.join(BASE, 'FLAG2027'); FEATDIR = os.path.join(PROJECT, 'features'); RESDIR = os.path.join(PROJECT, 'results'); CACHE = os.path.join(PROJECT, 'models_cache')
ZIP_DIR = json.load(open(os.path.join(PROJECT, 'paths.json')))['ZIP_DIR']
MAV_ZIPS, MAV_LISTS = os.path.join(BASE, 'dataset'), os.path.join(BASE, 'face_voice_association_splits')
SUMMARY = []
def note(msg):
    print(msg); SUMMARY.append(msg)
IMG = ('.jpg', '.jpeg', '.png')
FULL = '/content/flag_faces'; os.makedirs(FULL, exist_ok=True)
for z in ['train_set.zip', 'dev_set.zip']:
    if not os.path.exists(os.path.join(FULL, z + '.done')):
        with zipfile.ZipFile(os.path.join(ZIP_DIR, z)) as zf:
            zf.extractall(FULL, members=[m for m in zf.namelist() if m.lower().endswith(IMG + ('.txt', '.csv'))])
        open(os.path.join(FULL, z + '.done'), 'w').close()
TXT = [os.path.join(r, f) for r, _, fs in os.walk(FULL) for f in fs if f.endswith(('.txt', '.csv'))]
def find_one(*parts):
    hits = [p for p in TXT if all(x in p for x in parts)]
    assert len(hits) == 1, 'looking for %s -> found %s' % (parts, hits)
    return hits[0]
def resolve(rel, list_path):
    q = os.path.dirname(list_path)
    while q.startswith(FULL):
        c = os.path.join(q, rel)
        if os.path.exists(c):
            return c
        q = os.path.dirname(q)
def face_token(tokens):
    t = [x for x in tokens if x.lower().endswith(IMG)]; assert len(t) == 1, tokens; return t[0]
SPLIT_NAMES = ['train', 'no_gender_English', 'no_gender_Bangla', 'gender_English', 'gender_Bangla']
FILES = {}
lp = find_one('train_English.txt'); FILES['train'] = [resolve(face_token(l.split()), lp) for l in open(lp) if l.strip()]
for s in SPLIT_NAMES[1:]:
    track, lang_ = s.rsplit('_', 1); lp = find_one('/' + track + '/', lang_ + '_test.txt')
    FILES[s] = [resolve(face_token(l.split()), lp) for l in open(lp) if l.strip()]
assert all(None not in v for v in FILES.values()), 'some face files missing - tell Claude'
def zip_index(zf):
    m = {}
    for n in zf.namelist():
        for key in ('voices/', 'faces/'):
            i = n.find(key)
            if i >= 0 and not n.endswith('/'):
                m[n[i:]] = n; break
    return m
MAVD = '/content/mav'; mav_paths, mspk = [], []
for ver, zname, lists in [('v1', 'mavceleb_v1_train.zip', ['English_train.txt', 'Urdu_train.txt']), ('v3', 'mavceleb_v3_tran.zip', ['English_train.txt', 'German_train.txt'])]:
    flag = os.path.join(MAVD, ver + '.faces.done'); need = not os.path.exists(flag)
    zf = zipfile.ZipFile(os.path.join(MAV_ZIPS, zname)); zi = zip_index(zf)
    for lf in lists:
        for l in open(os.path.join(MAV_LISTS, ver, lf)):
            t = l.split()
            if len(t) < 2:
                continue
            vm, fm = zi.get('voices/' + t[0]), zi.get('faces/' + t[1])
            if vm is None or fm is None:
                continue
            if need:
                zf.extract(fm, os.path.join(MAVD, ver))
            mav_paths.append(os.path.join(MAVD, ver, fm)); mspk.append(ver + '_' + t[0].split('/')[0])
    zf.close()
    if need:
        open(flag, 'w').close()
MAV = dict(np.load(os.path.join(FEATDIR, 'mavceleb_v1v3.npz')))
assert list(MAV['spk']) == mspk, 'MAV v1/v3 row order differs - tell Claude'
FILES['mav'] = mav_paths
EXTRA = dict(np.load(os.path.join(FEATDIR, 'mavceleb_extra.npz')))
MAX_PER_ID = 120
VPAT = re.compile(r'voices/(id\d+)/([^/]+)/([^/]+)/([^/]+\.wav)$'); FPAT = re.compile(r'faces/(id\d+)/([^/]+)/([^/]+)/([^/]+\.jpg)$')
v1_used = set()
for lf in ['English_train.txt', 'Urdu_train.txt']:
    for l in open(os.path.join(MAV_LISTS, 'v1', lf)):
        t = l.split()
        if t:
            v1_used.add(t[0].split('/')[0])
rows = []
for ver, zname, skip_ids in [('v2', 'mavceleb_v2_complete.zip', set()), ('v1', 'mavceleb_v1_complete.zip', v1_used)]:
    D = '/content/mavx/' + ver; flag = os.path.join(D, 'faces.done'); need = not os.path.exists(flag)
    zf = zipfile.ZipFile(os.path.join(MAV_ZIPS, zname)); voices, faces = {}, {}
    for n in zf.namelist():
        mv = VPAT.search(n)
        if mv:
            voices.setdefault(mv.group(1), []).append((mv.group(2), mv.group(3), n)); continue
        mf = FPAT.search(n)
        if mf:
            faces.setdefault((mf.group(1), mf.group(2), mf.group(3)), []).append(n)
    ids_ = sorted(i for i in voices if i not in skip_ids); rng = np.random.default_rng(0)
    for pid in ids_:
        vs = voices[pid]; vs = [vs[j] for j in rng.permutation(len(vs))[:MAX_PER_ID]]
        for lang_, vid, vn in vs:
            fr = faces.get((pid, lang_, vid)) or [f for (p, l, v), fl in faces.items() if p == pid for f in fl]
            if not fr:
                continue
            fn = fr[rng.integers(len(fr))]
            if need:
                zf.extract(fn, D)
            rows.append((ver + '_' + pid, os.path.join(D, fn)))
    zf.close()
    if need:
        open(flag, 'w').close()
spk_all = np.array([r[0] for r in rows]); keep = np.ones(len(rows), bool)
if len(rows) != len(EXTRA['spk']):
    j = 0; keep[:] = False
    for i in range(len(rows)):
        if j < len(EXTRA['spk']) and spk_all[i] == EXTRA['spk'][j]:
            keep[i] = True; j += 1
    assert j == len(EXTRA['spk']), 'could not align MAV extra rows - tell Claude'
FILES['extra'] = [r[1] for r, k in zip(rows, keep) if k]
# alignment check: recompute CLIP ViT-L/14 on 100 random rows per set and compare with the cached clip features
from transformers import CLIPModel, CLIPProcessor
_cm = CLIPModel.from_pretrained('openai/clip-vit-large-patch14').half().to(dev).eval(); _cp = CLIPProcessor.from_pretrained('openai/clip-vit-large-patch14')
def clip_ref(paths):
    x = _cp(images=[Image.open(p).convert('RGB') for p in paths], return_tensors='pt')['pixel_values'].to(dev, torch.float16)
    with torch.no_grad():
        return _cm.visual_projection(_cm.vision_model(pixel_values=x).pooler_output).float().cpu().numpy()
CLIPF = np.load(os.path.join(FEATDIR, 'clip.npz'))
rng = np.random.default_rng(1)
for k, saved in {'train': CLIPF['train'], 'mav': MAV['clip'], 'extra': EXTRA['clip']}.items():
    idx = rng.choice(len(saved), 100, replace=False); new = clip_ref([FILES[k][i] for i in idx]); old = saved[idx]
    cos = (new * old).sum(1) / (np.linalg.norm(new, axis=1) * np.linalg.norm(old, axis=1) + 1e-9)
    note('face alignment %-6s median cosine %.4f' % (k, np.median(cos))); assert np.median(cos) > 0.98, 'face rows misaligned - tell Claude'
del _cm; torch.cuda.empty_cache()
note('Faces: ' + ', '.join('%s %d' % (k, len(v)) for k, v in FILES.items()))

In [ ]:
# Cell 3 [v2 29-Oct] - the four encoders. Fixes: (1) model downloads go to the LOCAL disk (4-GB files written to Google Drive came back corrupt:
# 'header too large'); (2) DFN falls back to the transformers loader if open_clip rejects its .bin; (3) FaRL has no processor config -> use the
# standard CLIP ViT-B/16 one (FaRL is a CLIP ViT-B/16 with the same preprocessing).
import open_clip, shutil
LOCAL_CACHE = '/content/hf_cache'; os.makedirs(LOCAL_CACHE, exist_ok=True)
os.environ['HF_HOME'] = LOCAL_CACHE; os.environ['HF_HUB_CACHE'] = os.path.join(LOCAL_CACHE, 'hub')
bad = os.path.join(CACHE, 'open_clip')
if os.path.isdir(bad):
    shutil.rmtree(bad, ignore_errors=True)          # remove the corrupt Drive copies so nothing picks them up
def run_batches(paths, bs, prep, fwd, desc):
    out = []
    for i in tqdm(range(0, len(paths), bs), desc=desc):
        x = prep([Image.open(p).convert('RGB') for p in paths[i:i + bs]])
        with torch.no_grad():
            out.append(fwd(x).float().cpu().numpy())
    return np.concatenate(out)
def make_openclip(hub_id, bs=32):
    def fn(paths):
        model, _, pre = open_clip.create_model_and_transforms('hf-hub:' + hub_id, cache_dir=os.path.join(LOCAL_CACHE, 'open_clip'))
        model = model.half().to(dev).eval()
        emb = run_batches(paths, bs, lambda ims: torch.stack([pre(im) for im in ims]).to(dev, torch.float16), model.encode_image, hub_id.split('/')[-1])
        del model; torch.cuda.empty_cache(); return emb
    return fn
def make_hf_clip(name, proc_name=None, bs=32):
    def fn(paths):
        model = CLIPModel.from_pretrained(name, cache_dir=LOCAL_CACHE).half().to(dev).eval(); proc = CLIPProcessor.from_pretrained(proc_name or name, cache_dir=LOCAL_CACHE)
        emb = run_batches(paths, bs, lambda ims: proc(images=ims, return_tensors='pt')['pixel_values'].to(dev, torch.float16), lambda x: model.get_image_features(pixel_values=x), name.split('/')[-1])
        del model; torch.cuda.empty_cache(); return emb
    return fn
def enc_dfn(paths):
    try:
        return make_openclip('apple/DFN5B-CLIP-ViT-H-14-378', bs=16)(paths)
    except Exception as e:
        print('open_clip could not load DFN (%r) -> transformers loader' % repr(e)[:120]); torch.cuda.empty_cache()
        return make_hf_clip('apple/DFN5B-CLIP-ViT-H-14-378', bs=16)(paths)
ENCODERS = {'vith': make_openclip('laion/CLIP-ViT-H-14-laion2B-s32B-b79K'),
            'dfn': enc_dfn,
            'pe': make_openclip('timm/PE-Core-L-14-336'),
            'farl': make_hf_clip('Green-Sky/FaRL-Base-Patch16-LAIONFace20M-ep64', proc_name='openai/clip-vit-base-patch16', bs=64)}
ORDER = SPLIT_NAMES + ['mav', 'extra']
allp = [p for s in ORDER for p in FILES[s]]
def split_back(emb):
    out, i = {}, 0
    for s in ORDER:
        n = len(FILES[s]); out[s] = emb[i:i + n].astype(np.float32); i += n
    return out
for name, fn in ENCODERS.items():
    p = os.path.join(FEATDIR, 'face_' + name + '.npz')
    if os.path.exists(p):
        print(name, 'already extracted'); continue
    t0 = time.time()
    try:
        emb = fn(allp); np.savez(p, **split_back(emb)); note('%-5s dim %4d | %d faces in %.1f min' % (name, emb.shape[1], len(allp), (time.time() - t0) / 60))
    except Exception as e:
        note('ENCODER %s FAILED: %r' % (name, repr(e)[:300])); torch.cuda.empty_cache()
    shutil.rmtree(os.path.join(LOCAL_CACHE, 'open_clip'), ignore_errors=True); shutil.rmtree(os.path.join(LOCAL_CACHE, 'hub'), ignore_errors=True)   # free the local disk between 4-GB models
print('[v2 29-Oct] extraction cell done')

In [ ]:
# Cell 4 [v2 29-Oct] - CV: each encoder replacing CLIP, then CLIP + best concatenated (challenge + all MAV, R4 voice stack, prototypes; 5-fold x 3 seeds)
import torch.nn as nn, torch.nn.functional as Fn
from sklearn.metrics import roc_curve
def read_feat(path, dim):
    df = pd.read_csv(path, header=None)
    return df.iloc[:, :dim].values.astype(np.float32), (df.iloc[:, dim].values if df.shape[1] > dim else None)
Xv, y = read_feat(find_one('train_English_voices.csv'), 192); y = y.astype(int)
FEAT = {'ecapa': {'train': Xv, 'mav': MAV['ecapa'], 'extra': EXTRA['ecapa']}}
for s in SPLIT_NAMES[1:]:
    track, lang_ = s.rsplit('_', 1); FEAT['ecapa'][s] = read_feat(find_one('/' + track + '/', lang_ + '_test_voices.csv'), 192)[0]
for name in ['clip', 'wavlm_sv', 'r293', 'redim', 'audeer']:
    d = np.load(os.path.join(FEATDIR, name + '.npz')); FEAT[name] = {s: d[s] for s in d.files if not s.endswith('_ag')}
FEAT['clip']['mav'], FEAT['clip']['extra'] = MAV['clip'], EXTRA['clip']; FEAT['wavlm_sv']['mav'], FEAT['wavlm_sv']['extra'] = MAV['wavlm_sv'], EXTRA['wavlm_sv']
FEAT['r293']['extra'] = EXTRA['r293']
for name in ENCODERS:
    p = os.path.join(FEATDIR, 'face_' + name + '.npz')
    if os.path.exists(p):
        d = np.load(p); FEAT[name] = {s: d[s] for s in ORDER}
assert any(n in FEAT for n in ENCODERS), 'no new face encoder was extracted - paste the ENCODER ... FAILED lines of Cell 3 to Claude'
train_lines = [l.split() for l in open(find_one('train_English.txt')) if l.strip()]
cls2spk = {int(l[5]): l[4] for l in train_lines}
meta = pd.read_csv(os.path.join(ZIP_DIR, 'meta_file_train_set.csv'), header=None, names=['spk', 'g'])
spk2g = dict(zip(meta.spk.astype(str).str.strip(), meta.g.astype(str).str.strip().str.lower()))
ids = np.unique(y); id_gender = {c: spk2g[cls2spk[c]] for c in ids}
N_CH = len(y); ext_spk = list(MAV['spk']) + list(EXTRA['spk'])
ext_lab = {s: 1000 + i for i, s in enumerate(sorted(set(ext_spk)))}
Y_ALL = np.r_[y, np.array([ext_lab[s] for s in ext_spk])]; D_ALL = np.r_[np.zeros(N_CH, int), np.ones(len(ext_spk), int)]; EXT = N_CH + np.arange(len(ext_spk))
def l2(X):
    return X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-9)
def build(names, split):
    return np.concatenate([l2(FEAT[n][split]) for n in names], 1).astype(np.float32)
def build_train(names):
    return np.concatenate([build(names, 'train'), build(names, 'mav'), build(names, 'extra')])
def eer(labels, scores):
    fpr, tpr, _ = roc_curve(labels, scores); fnr = 1 - tpr
    i = np.nanargmin(np.abs(fnr - fpr)); return (fpr[i] + fnr[i]) / 2 * 100
def make_pairs(idx, n=2000, same_gender_neg=False, seed=0):
    rng = np.random.default_rng(seed); spk_ = np.unique(y[idx]); rows_ = {s: idx[y[idx] == s] for s in spk_}
    V, F, L = [], [], []
    for _ in range(n):
        s = rng.choice(spk_); V.append(rng.choice(rows_[s])); F.append(rng.choice(rows_[s])); L.append(1)
        s = rng.choice(spk_)
        cand = [t for t in spk_ if t != s and (not same_gender_neg or id_gender[t] == id_gender[s])] or [t for t in spk_ if t != s]
        t = rng.choice(cand); V.append(rng.choice(rows_[s])); F.append(rng.choice(rows_[t])); L.append(0)
    return np.array(V), np.array(F), np.array(L)
K = 5; rng = np.random.default_rng(42); folds = [[] for _ in range(K)]
for gg in sorted(set(id_gender.values())):
    members = [c for c in ids if id_gender[c] == gg]; rng.shuffle(members)
    for i, c in enumerate(members):
        folds[i % K].append(c)
FOLD_DATA = []
for k in range(K):
    val_ids = set(folds[k]); va_idx = np.where([c in val_ids for c in y])[0]; tr_idx = np.where([c not in val_ids for c in y])[0]
    FOLD_DATA.append((tr_idx, [make_pairs(va_idx, seed=k), make_pairs(va_idx, same_gender_neg=True, seed=100 + k)]))
_PROTO = {}
def max_sep_prototypes(n):
    if n in _PROTO:
        return _PROTO[n]
    P = np.array([[1.0, -1.0]])
    for k in range(2, n):
        Q = np.zeros((k, k + 1)); Q[0, 0] = 1.0; Q[0, 1:] = -1.0 / k; Q[1:, 1:] = np.sqrt(1 - 1.0 / k ** 2) * P; P = Q
    _PROTO[n] = torch.tensor(P.T, dtype=torch.float32); return _PROTO[n]
class Branch(nn.Module):
    def __init__(self, d_in, d=128):
        super().__init__(); self.net = nn.Sequential(nn.Linear(d_in, d), nn.BatchNorm1d(d), nn.ReLU(), nn.Dropout(0.5))
    def forward(self, x):
        return Fn.normalize(self.net(x), dim=1)
class FOP(nn.Module):
    def __init__(self, n_class, d_face, d_voice, d=128):
        super().__init__()
        self.face, self.voice = Branch(d_face, d), Branch(d_voice, d)
        self.att = nn.Sequential(nn.Linear(2 * d, d), nn.BatchNorm1d(d), nn.ReLU(), nn.Linear(d, d))
        self.cls = nn.Linear(d, n_class - 1); self.register_buffer('P', max_sep_prototypes(n_class))
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight); nn.init.constant_(m.bias, 0.01)
    def embed(self, xf, xv):
        f, v = self.face(xf), self.voice(xv); return torch.tanh(f), torch.tanh(v), f, v
    def forward(self, xf, xv):
        ft, vt, f, v = self.embed(xf, xv); a = torch.sigmoid(self.att(torch.cat([f, v], 1))); fused = ft * a + (1 - a) * vt
        return fused, self.cls(fused) @ self.P.t()
def opl_loss(feat, lab):
    feat = Fn.normalize(feat, dim=1); dot = feat @ feat.t(); same = (lab[:, None] == lab[None, :]).float()
    pos = same - torch.eye(len(lab), device=feat.device); neg = 1 - same
    return (1 - (pos * dot).sum() / (pos.sum() + 1e-6)) + 0.7 * (neg * dot.abs()).sum() / (neg.sum() + 1e-6)
@torch.no_grad()
def same_score(model, xf, xv):
    model.eval(); mf, sf_, mv, sv = model.norm; ft, vt, _, _ = model.embed((xf - mf) / sf_, (xv - mv) / sv)
    return -(ft - vt).norm(dim=1).cpu().numpy()
def dnorm(F_raw, V_raw, tr_idx):
    F, V = F_raw.copy(), V_raw.copy(); ch = None
    for d in np.unique(D_ALL[tr_idx]):
        trd = tr_idx[D_ALL[tr_idx] == d]; alld = np.where(D_ALL == d)[0]
        st = (F_raw[trd].mean(0), F_raw[trd].std(0) + 1e-6, V_raw[trd].mean(0), V_raw[trd].std(0) + 1e-6)
        F[alld] = (F_raw[alld] - st[0]) / st[1]; V[alld] = (V_raw[alld] - st[2]) / st[3]
        if d == 0:
            ch = st
    return F, V, ch
SEEDS = [1, 2, 3]; P_ID, K_CLIP, ALPHA, LR, MAX_EP, PATIENCE, STEPS = 16, 8, 2.0, 1e-4, 100, 15, 60
def train_combo(F_raw, V_raw, tr_idx, vsets=None, seed=1, max_epochs=MAX_EP):
    torch.manual_seed(seed); rng = np.random.default_rng(seed); all_tr = np.concatenate([tr_idx, EXT])
    F, V, ch = dnorm(F_raw, V_raw, all_tr); XF = torch.tensor(F, device=dev); XV = torch.tensor(V, device=dev)
    cl = np.unique(Y_ALL[all_tr]); rm = {c: i for i, c in enumerate(cl)}
    ytr_np = np.array([rm[c] for c in Y_ALL[all_tr]]); ytr = torch.tensor(ytr_np, device=dev); pos_by_cls = [np.where(ytr_np == c)[0] for c in range(len(cl))]
    tr = torch.tensor(all_tr, device=dev); model = FOP(len(cl), XF.shape[1], XV.shape[1]).to(dev)
    model.norm = (XF[tr].mean(0), XF[tr].std(0) + 1e-6, XV[tr].mean(0), XV[tr].std(0) + 1e-6); mf, sf_, mv, sv = model.norm
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=0.01); best, best_ep, best_state, wait = 1e9, max_epochs, None, 0
    for ep in range(1, max_epochs + 1):
        model.train()
        for _ in range(STEPS):
            chosen = rng.choice(len(cl), P_ID, replace=False)
            j = torch.tensor(np.concatenate([rng.choice(pos_by_cls[c], K_CLIP, replace=len(pos_by_cls[c]) < K_CLIP) for c in chosen]), device=dev)
            fused, logits = model((XF[tr[j]] - mf) / sf_, (XV[tr[j]] - mv) / sv)
            loss = Fn.cross_entropy(logits, ytr[j]) + ALPHA * opl_loss(fused, ytr[j]); opt.zero_grad(); loss.backward(); opt.step()
        if vsets is None:
            continue
        score = np.mean([eer(L, same_score(model, XF[Fi], XV[Vi])) for Vi, Fi, L in vsets])
        if score < best - 0.05:
            best, best_ep, best_state, wait = score, ep, copy.deepcopy(model.state_dict()), 0
        else:
            wait += 1
            if wait >= PATIENCE:
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best_ep, XF, XV, ch
R4 = ['ecapa', 'wavlm_sv', 'r293', 'redim', 'audeer']
CSV = os.path.join(RESDIR, 'nb23_cv.csv')
# reference F0 = CLIP alone: identical configuration to NB21's S0 -> reuse those 15 runs if present
if not os.path.exists(CSV) and os.path.exists(os.path.join(RESDIR, 'nb21_cv.csv')):
    r21 = pd.read_csv(os.path.join(RESDIR, 'nb21_cv.csv')); r21 = r21[r21.combo == 'S0_R4_ref'].copy(); r21['combo'] = 'F0_clip'; r21.to_csv(CSV, index=False)
done = pd.read_csv(CSV) if os.path.exists(CSV) else pd.DataFrame(columns=['combo', 'seed', 'fold']); t0 = time.time()
def run_combos(COMBOS):
    global done
    for name, (fn_, vn_) in COMBOS.items():
        F_raw, V_raw = build_train(fn_), build_train(vn_)
        for seed in SEEDS:
            for k in range(K):
                if ((done.combo == name) & (done.seed == seed) & (done.fold == k)).any():
                    continue
                tr_idx, vsets = FOLD_DATA[k]; model, best_ep, XF, XV, _ = train_combo(F_raw, V_raw, tr_idx, vsets, seed=seed)
                e_std, e_gen = [eer(L, same_score(model, XF[Fi], XV[Vi])) for Vi, Fi, L in vsets]
                row = pd.DataFrame([dict(combo=name, seed=seed, fold=k, best_ep=best_ep, std=e_std, gender=e_gen)])
                row.to_csv(CSV, mode='a', header=not os.path.exists(CSV), index=False); done = pd.concat([done, row])
                print('%-14s seed %d fold %d | ep %3d | std %.2f | gender %.2f | %.0f min' % (name, seed, k, best_ep, e_std, e_gen, (time.time() - t0) / 60))
def table():
    R = pd.read_csv(CSV); R['overall'] = (R['std'] + R['gender']) / 2
    return R, R.groupby(['combo', 'seed'])[['std', 'gender', 'overall']].mean().groupby('combo').mean().round(2)
STAGE1 = {'F0_clip': (['clip'], R4)}
STAGE1.update({'F1_' + n: ([n], R4) for n in ENCODERS if n in FEAT})
run_combos(STAGE1); R, tab = table()
singles = [c for c in tab.index if c.startswith('F1_')]
STAGE2 = {}
if singles:
    bname = tab.loc[singles, 'overall'].idxmin()[3:]
    STAGE2['F2_clip+' + bname] = (['clip', bname], R4)
    if bname != 'vith' and 'vith' in FEAT:
        STAGE2['F2_clip+vith'] = (['clip', 'vith'], R4)
    run_combos(STAGE2); R, tab = table()
note('FACE ENCODERS, voice = R4 stack (challenge + all MAV, prototypes; 5-fold x 3 seeds)'); note(tab.to_string())
BEST = tab['overall'].idxmin(); COMBOS = {**STAGE1, **STAGE2}
note('BEST: %s (%.2f) vs CLIP reference %.2f' % (BEST, tab.loc[BEST, 'overall'], tab.loc['F0_clip', 'overall']))

In [ ]:
# Cell 5 - ZIP for the best combo (no VoxCeleb) + saved dev scores
LANG_FILE = {'English': 'sub_score_v4_English_heard.txt', 'Bangla': 'sub_score_v4_Bangla_unheard.txt'}
SUBDIR = os.path.join(PROJECT, 'submissions'); os.makedirs(SUBDIR, exist_ok=True)
fn_, vn_ = COMBOS[BEST]; F_raw, V_raw = build_train(fn_), build_train(vn_)
ep = int(np.median(R[R.combo == BEST]['best_ep']))
finals = [train_combo(F_raw, V_raw, np.arange(N_CH), None, seed=s, max_epochs=ep)[0] for s in SEEDS]
_, _, ch = dnorm(F_raw, V_raw, np.concatenate([np.arange(N_CH), EXT]))
sc = {}
for track in ['no_gender', 'gender']:
    for lang_ in LANG_FILE:
        split = track + '_' + lang_
        xf = torch.tensor(((build(fn_, split) - ch[0]) / ch[1]).astype(np.float32), device=dev); xv = torch.tensor(((build(vn_, split) - ch[2]) / ch[3]).astype(np.float32), device=dev)
        sc[split] = np.mean([same_score(m, xf, xv) for m in finals], axis=0)
tag = BEST.replace('+', '_'); OUT = '/content/sub_23_' + tag
for track in ['no_gender', 'gender']:
    os.makedirs(os.path.join(OUT, track), exist_ok=True)
    for lang_, fname in LANG_FILE.items():
        split = track + '_' + lang_; pair_ids = [l.split()[0] for l in open(find_one('/' + track + '/', lang_ + '_test.txt')) if l.strip()]
        assert len(pair_ids) == len(sc[split]) and np.isfinite(sc[split]).all()
        with open(os.path.join(OUT, track, fname), 'w') as f:
            for pid, v in zip(pair_ids, -sc[split]):
                f.write('%s %.6f\n' % (pid, v))
zp = os.path.join(SUBDIR, '23_%s_noVox.zip' % tag)
with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as zf:
    for track in ['no_gender', 'gender']:
        for fname in LANG_FILE.values():
            zf.write(os.path.join(OUT, track, fname), arcname=track + '/' + fname)
np.savez(os.path.join(RESDIR, 'devscores_23_%s.npz' % tag), **sc); note('Submission ZIP saved: ' + zp[len(BASE):])
print('=' * 70); print('SUMMARY  (copy everything between the lines and paste to Claude)'); print('=' * 70)
for s in SUMMARY:
    print(s)
print('=' * 70)